In [12]:
import random
import numpy as np
from collections import deque

class MazeMDP:
    def __init__(self, N, start, goal, walls):
        self.N = N
        self.start = start
        self.goal = goal
        self.walls = set(walls)
        self.actions = ["UP", "DOWN", "LEFT", "RIGHT"]
        self.reset()
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        x, y = self.state
        if action == "UP":
            nxt = (x-1, y)
        elif action == "DOWN":
            nxt = (x+1, y)
        elif action == "LEFT":
            nxt = (x, y-1)
        elif action == "RIGHT":
            nxt = (x, y+1)
        else:
            nxt = (x, y)
        
        # check boundaries and walls
        if (nxt[0] < 0 or nxt[0] >= self.N or nxt[1] < 0 or nxt[1] >= self.N or nxt in self.walls):
            nxt = self.state
            reward = -5  # penalty for hitting wall/border
        elif nxt == self.goal:
            reward = 10
        else:
            reward = -1
        
        self.state = nxt
        done = (nxt == self.goal)
        return nxt, reward, done

    def random_action(self):
        return random.choice(self.actions)

    def greedy_action(self, state):
        # Manhattan distance heuristic policy
        x, y = state
        gx, gy = self.goal
        candidates = []
        if gx < x:
            candidates.append("UP")
        if gx > x:
            candidates.append("DOWN")
        if gy < y:
            candidates.append("LEFT")
        if gy > y:
            candidates.append("RIGHT")
        # try candidates first
        for a in candidates:
            x, y = state
            if a == "UP": nxt = (x-1, y)
            elif a == "DOWN": nxt = (x+1, y)
            elif a == "LEFT": nxt = (x, y-1)
            else: nxt = (x, y+1)
            if nxt[0] >= 0 and nxt[0] < self.N and nxt[1] >= 0 and nxt[1] < self.N and nxt not in self.walls:
                return a
        return self.random_action()

class SmartMazePolicy:
    """
    A custom policy that combines multiple strategies:
    1. A* pathfinding for optimal route planning
    2. Memory of visited states to avoid loops
    3. Wall avoidance heuristics
    4. Adaptive exploration when stuck
    """
    
    def __init__(self, env):
        self.env = env
        self.visited_states = set()
        self.visit_count = {}
        self.last_positions = deque(maxlen=5)  # Track recent positions
        self.stuck_counter = 0
        
    def reset(self):
        """Reset policy state for new episode"""
        self.visited_states.clear()
        self.visit_count.clear()
        self.last_positions.clear()
        self.stuck_counter = 0
    
    def manhattan_distance(self, state1, state2):
        """Calculate Manhattan distance between two states"""
        return abs(state1[0] - state2[0]) + abs(state1[1] - state2[1])
    
    def get_valid_neighbors(self, state):
        """Get all valid neighboring states"""
        x, y = state
        neighbors = []
        actions = ["UP", "DOWN", "LEFT", "RIGHT"]
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        
        for action, (dx, dy) in zip(actions, moves):
            nx, ny = x + dx, y + dy
            if (0 <= nx < self.env.N and 
                0 <= ny < self.env.N and 
                (nx, ny) not in self.env.walls):
                neighbors.append(((nx, ny), action))
        return neighbors
    
    def a_star_path(self, start, goal):
        """A* pathfinding algorithm"""
        from heapq import heappush, heappop
        
        open_set = [(0, start, [])]
        closed_set = set()
        
        while open_set:
            f_score, current, path = heappop(open_set)
            
            if current == goal:
                return path
            
            if current in closed_set:
                continue
            
            closed_set.add(current)
            
            for neighbor, action in self.get_valid_neighbors(current):
                if neighbor in closed_set:
                    continue
                
                g_score = len(path) + 1
                h_score = self.manhattan_distance(neighbor, goal)
                f_score = g_score + h_score
                
                new_path = path + [action]
                heappush(open_set, (f_score, neighbor, new_path))
        
        return []  # No path found
    
    def is_stuck(self, current_state):
        """Detect if agent is stuck in a loop"""
        self.last_positions.append(current_state)
        
        # Check if we're revisiting recent positions frequently
        if len(self.last_positions) == 5:
            if len(set(self.last_positions)) <= 2:
                return True
        
        # Check visit frequency
        visit_freq = self.visit_count.get(current_state, 0)
        return visit_freq > 3
    
    def exploration_action(self, current_state):
        """Choose action for exploration when stuck"""
        valid_neighbors = self.get_valid_neighbors(current_state)
        
        if not valid_neighbors:
            return random.choice(self.env.actions)
        
        # Prefer less visited states
        best_neighbors = []
        min_visits = float('inf')
        
        for (nx, ny), action in valid_neighbors:
            visits = self.visit_count.get((nx, ny), 0)
            if visits < min_visits:
                min_visits = visits
                best_neighbors = [action]
            elif visits == min_visits:
                best_neighbors.append(action)
        
        return random.choice(best_neighbors)
    
    def choose_action(self, current_state):
        """Main policy decision function"""
        # Update visit tracking
        self.visit_count[current_state] = self.visit_count.get(current_state, 0) + 1
        self.visited_states.add(current_state)
        
        # Check if we're stuck
        if self.is_stuck(current_state):
            self.stuck_counter += 1
            if self.stuck_counter > 2:  # If stuck for multiple steps
                return self.exploration_action(current_state)
        else:
            self.stuck_counter = 0
        
        # Try A* pathfinding first
        optimal_path = self.a_star_path(current_state, self.env.goal)
        if optimal_path:
            return optimal_path[0]
        
        # Fallback to greedy with visit penalty
        x, y = current_state
        gx, gy = self.env.goal
        
        # Generate candidate actions based on goal direction
        candidates = []
        if gx < x:
            candidates.append("UP")
        if gx > x:
            candidates.append("DOWN")
        if gy < y:
            candidates.append("LEFT")
        if gy > y:
            candidates.append("RIGHT")
        
        # Evaluate candidates considering visit history
        best_actions = []
        best_score = float('inf')
        
        for action in candidates:
            # Check if action is valid
            x, y = current_state
            if action == "UP":
                nxt = (x-1, y)
            elif action == "DOWN":
                nxt = (x+1, y)
            elif action == "LEFT":
                nxt = (x, y-1)
            else:  # RIGHT
                nxt = (x, y+1)
            
            # Skip invalid moves
            if (nxt[0] < 0 or nxt[0] >= self.env.N or 
                nxt[1] < 0 or nxt[1] >= self.env.N or 
                nxt in self.env.walls):
                continue
            
            # Score based on distance to goal and visit frequency
            distance_score = self.manhattan_distance(nxt, self.env.goal)
            visit_penalty = self.visit_count.get(nxt, 0) * 2
            total_score = distance_score + visit_penalty
            
            if total_score < best_score:
                best_score = total_score
                best_actions = [action]
            elif total_score == best_score:
                best_actions.append(action)
        
        if best_actions:
            return random.choice(best_actions)
        
        # Final fallback - random valid action
        return self.exploration_action(current_state)


# Extension to your existing MazeMDP class
def add_smart_policy_to_maze(env):
    """
    Add the smart policy to your existing MazeMDP environment
    Usage: 
        env = MazeMDP(N, start, goal, walls)
        add_smart_policy_to_maze(env)
        action = env.smart_action(current_state)
    """
    env.smart_policy = SmartMazePolicy(env)
    
    def smart_action(state):
        return env.smart_policy.choose_action(state)
    
    def reset_with_policy():
        state = env.reset()
        env.smart_policy.reset()
        return state
    
    # Add methods to environment
    env.smart_action = smart_action
    env.reset_with_policy = reset_with_policy
    
    return env

N = 5
start = (1, 0)
goal = (0, 4)
walls = [(3, 1), (1, 2), (3, 3), (2, 2)]

# Example usage with your existing code:
if __name__ == "__main__":
    # This shows how to integrate with your existing MazeMDP
    
    # Your existing environment setup (copy from your notebook)
    env = MazeMDP(N, start, goal, walls)
    
    # Add smart policy
    env = add_smart_policy_to_maze(env)

    # Modified run_episode function to include smart policy
    def run_episode_with_smart(env, policy="random", max_steps=50):
        if policy == "smart":
            state = env.reset_with_policy()  # Reset both env and policy
        else:
            state = env.reset()
        
        trajectory = [state]
        total_reward, steps = 0, 0
        
        for _ in range(max_steps):
            if policy == "random":
                action = env.random_action()
            elif policy == "greedy":
                action = env.greedy_action(state)
            elif policy == "smart":
                action = env.smart_action(state)
            else:
                action = env.random_action()
            
            nxt, reward, done = env.step(action)
            trajectory.append(nxt)
            total_reward += reward
            steps += 1
            state = nxt
            
            if done:
                break
        
        return trajectory, steps, total_reward
    
    def run_experiments(env, policy, episodes=1000000):
        stats = {"steps": [], "rewards": []}
        sample_traj = None
        for i in range(episodes):
            traj, steps, rew = run_episode_with_smart(env, policy)
            if i == 0:
                sample_traj = traj
            stats["steps"].append(steps)
            stats["rewards"].append(rew)
        return sample_traj, stats

traj_smart, stats_smart = run_experiments(env, "smart")

print("Smart sample trajectory:", traj_smart)

print("Smart avg steps:", np.mean(stats_smart['steps']), "avg reward:", np.mean(stats_smart['rewards']))

Smart sample trajectory: [(1, 0), (0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
Smart avg steps: 5.0 avg reward: 6.0
